# Biomarker Overlap Analysis: Matched vs Unmatched & ATE vs noIPTW

Compare FDR-significant markers across:
1. **Cohort overlap** — `cohort1` (first-line unmatched) vs `cohort2` (1:1 line-matched) within each (ps_model, weight_type, cancer_type)
2. **Weighting overlap (Track 2)** — `ATE` vs `noIPTW` within each (cohort, ps_model, cancer_type)
3. **Weighting overlap (Track 1)** — `ATE` vs `unweighted` within each (cohort, ps_model, cancer_type)

For overlapping markers, we check HR direction concordance and effect size stability.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from matplotlib.patches import Patch
import seaborn as sns

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "figure.dpi": 150,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
})

# ── Paths ─────────────────────────────────────────────────────────────
DATA_PATH = '/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/'
MARKER_PATH = os.path.join(DATA_PATH, 'biomarker_analysis/')
COMPILED_DIR = os.path.join(MARKER_PATH, 'compiled_results/')
FIGURE_PATH = '/data/gusev/USERS/jpconnor/figures/clinical_text_embedding_project/'
OVERLAP_FIG_PATH = os.path.join(FIGURE_PATH, 'biomarker_analysis/overlap_analysis/')
os.makedirs(OVERLAP_FIG_PATH, exist_ok=True)

# ── Load compiled results ─────────────────────────────────────────────
t1 = pd.read_csv(os.path.join(COMPILED_DIR, 'track1_all_significant_hits.csv'))
t2 = pd.read_csv(os.path.join(COMPILED_DIR, 'track2_all_significant_hits.csv'))

print(f"Track 1: {len(t1)} significant hits, {t1['marker'].nunique()} unique markers")
print(f"Track 2: {len(t2)} significant hits, {t2['marker'].nunique()} unique markers")
print(f"\nTrack 1 — cohorts: {sorted(t1['cohort'].unique())}, "
      f"weights: {sorted(t1['weight_type'].unique())}")
print(f"Track 2 — cohorts: {sorted(t2['cohort'].unique())}, "
      f"weights: {sorted(t2['weight_type'].unique())}")

In [ ]:
# ── Overlap utilities ─────────────────────────────────────────────────

def compute_overlap(df, group_col, val_a, val_b, fixed_cols, marker_col='marker',
                    hr_col=None):
    """Compute marker set overlap between two values of group_col.

    For each unique combination of fixed_cols, finds markers significant in
    val_a, val_b, or both. Optionally checks HR direction concordance.

    Returns: DataFrame with one row per (fixed_cols combo, marker) that
    appears in at least one of val_a or val_b.
    """
    rows = []
    df_a = df[df[group_col] == val_a]
    df_b = df[df[group_col] == val_b]

    combos = df.groupby(fixed_cols).size().reset_index()[fixed_cols]
    # Deduplicate fixed_col combos present in either side
    combos_a = df_a[fixed_cols].drop_duplicates()
    combos_b = df_b[fixed_cols].drop_duplicates()
    combos = pd.merge(combos_a, combos_b, on=fixed_cols, how='outer')

    for _, combo in combos.iterrows():
        mask_a = pd.Series(True, index=df_a.index)
        mask_b = pd.Series(True, index=df_b.index)
        for col in fixed_cols:
            mask_a = mask_a & (df_a[col] == combo[col])
            mask_b = mask_b & (df_b[col] == combo[col])

        markers_a = set(df_a.loc[mask_a, marker_col])
        markers_b = set(df_b.loc[mask_b, marker_col])
        all_markers = markers_a | markers_b

        for m in sorted(all_markers):
            row = {c: combo[c] for c in fixed_cols}
            row['marker'] = m
            row[f'sig_in_{val_a}'] = m in markers_a
            row[f'sig_in_{val_b}'] = m in markers_b
            row['overlap'] = m in (markers_a & markers_b)

            if hr_col is not None:
                hr_a_rows = df_a.loc[mask_a & (df_a[marker_col] == m), hr_col]
                hr_b_rows = df_b.loc[mask_b & (df_b[marker_col] == m), hr_col]
                row[f'HR_{val_a}'] = float(hr_a_rows.iloc[0]) if len(hr_a_rows) else np.nan
                row[f'HR_{val_b}'] = float(hr_b_rows.iloc[0]) if len(hr_b_rows) else np.nan
                if not np.isnan(row[f'HR_{val_a}']) and not np.isnan(row[f'HR_{val_b}']):
                    row['direction_concordant'] = (
                        (row[f'HR_{val_a}'] > 1) == (row[f'HR_{val_b}'] > 1)
                    )
                else:
                    row['direction_concordant'] = np.nan

            rows.append(row)

    return pd.DataFrame(rows)


def overlap_summary(overlap_df, val_a, val_b):
    """Print summary statistics for an overlap DataFrame."""
    n_total = len(overlap_df)
    n_overlap = overlap_df['overlap'].sum()
    n_a_only = (overlap_df[f'sig_in_{val_a}'] & ~overlap_df[f'sig_in_{val_b}']).sum()
    n_b_only = (overlap_df[f'sig_in_{val_b}'] & ~overlap_df[f'sig_in_{val_a}']).sum()

    print(f"  Total unique markers: {n_total}")
    print(f"  {val_a}-only: {n_a_only}")
    print(f"  {val_b}-only: {n_b_only}")
    print(f"  Overlap (both): {n_overlap}")
    if n_total > 0:
        jaccard = n_overlap / n_total
        print(f"  Jaccard index: {jaccard:.3f}")

    if 'direction_concordant' in overlap_df.columns:
        concordant = overlap_df.loc[overlap_df['overlap'], 'direction_concordant']
        if len(concordant) > 0:
            n_conc = concordant.sum()
            print(f"  HR direction concordance (overlap): {n_conc}/{len(concordant)} "
                  f"({n_conc/len(concordant)*100:.0f}%)")

    return {'n_a_only': int(n_a_only), 'n_overlap': int(n_overlap),
            'n_b_only': int(n_b_only)}


print("Overlap utilities defined.")

## 1. Track 1: Matched (cohort2) vs Unmatched (cohort1)

For each (ps_model, weight_type, cancer_type), compare which markers are FDR-significant in cohort1 vs cohort2.

In [ ]:
# ── Track 1: cohort1 vs cohort2 overlap ───────────────────────────────

t1_cohort_overlap = compute_overlap(
    t1, group_col='cohort', val_a='cohort1', val_b='cohort2',
    fixed_cols=['ps_model', 'weight_type', 'cancer_type'],
    hr_col='HR_marker',
)

print("=" * 60)
print("Track 1: cohort1 (unmatched) vs cohort2 (matched)")
print("=" * 60)

# Per-stratum summaries
venn_data_t1_cohort = {}
for (ps, wt, ct), grp in t1_cohort_overlap.groupby(['ps_model', 'weight_type', 'cancer_type']):
    label = f"{ps} / {wt} / {ct}"
    print(f"\n--- {label} ---")
    counts = overlap_summary(grp, 'cohort1', 'cohort2')
    venn_data_t1_cohort[(ps, wt, ct)] = counts

# Aggregate across all strata
print(f"\n{'=' * 60}")
print("AGGREGATE (all strata)")
print("=" * 60)
_ = overlap_summary(t1_cohort_overlap, 'cohort1', 'cohort2')

In [ ]:
# ── Venn diagrams: Track 1 cohort overlap per stratum ─────────────────

strata = sorted(venn_data_t1_cohort.keys())
n_plots = len(strata)
ncols = min(n_plots, 3)
nrows = int(np.ceil(n_plots / ncols)) if n_plots > 0 else 1

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
if n_plots == 1:
    axes = np.array([axes])
axes = axes.flatten()

for i, (ps, wt, ct) in enumerate(strata):
    ax = axes[i]
    c = venn_data_t1_cohort[(ps, wt, ct)]
    v = venn2(
        subsets=(c['n_a_only'], c['n_b_only'], c['n_overlap']),
        set_labels=('cohort1\n(unmatched)', 'cohort2\n(matched)'),
        ax=ax,
    )
    ax.set_title(f"{ct}\n{ps} / {wt}", fontsize=10)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Track 1: Matched vs Unmatched Marker Overlap", fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(OVERLAP_FIG_PATH, 'track1_cohort_overlap_venns.png'))
plt.show()

In [ ]:
# ── HR concordance scatter: Track 1 cohort overlap ───────────────────

overlap_rows = t1_cohort_overlap[t1_cohort_overlap['overlap']].copy()

if len(overlap_rows) > 0:
    fig, ax = plt.subplots(figsize=(7, 7))

    # Use log2(HR) for symmetric visualization
    overlap_rows['logHR_cohort1'] = np.log2(overlap_rows['HR_cohort1'])
    overlap_rows['logHR_cohort2'] = np.log2(overlap_rows['HR_cohort2'])

    # Color by cancer type
    cancer_types = sorted(overlap_rows['cancer_type'].unique())
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(cancer_types), 1)))
    ct_color = dict(zip(cancer_types, colors))

    for ct in cancer_types:
        mask = overlap_rows['cancer_type'] == ct
        ax.scatter(
            overlap_rows.loc[mask, 'logHR_cohort1'],
            overlap_rows.loc[mask, 'logHR_cohort2'],
            label=ct, color=ct_color[ct], s=40, alpha=0.7, edgecolors='k', linewidth=0.5,
        )

    # Reference lines
    lims = [
        min(ax.get_xlim()[0], ax.get_ylim()[0]),
        max(ax.get_xlim()[1], ax.get_ylim()[1]),
    ]
    ax.plot(lims, lims, 'k--', alpha=0.4, linewidth=1, label='y=x')
    ax.axhline(0, color='grey', linewidth=0.5, linestyle=':')
    ax.axvline(0, color='grey', linewidth=0.5, linestyle=':')
    ax.set_xlim(lims)
    ax.set_ylim(lims)

    ax.set_xlabel('log2(HR) — cohort1 (unmatched)')
    ax.set_ylabel('log2(HR) — cohort2 (matched)')
    ax.set_title('Track 1: HR Concordance for Overlapping Markers')
    ax.legend(fontsize=9, loc='upper left')

    # Annotate concordance rate
    n_conc = overlap_rows['direction_concordant'].sum()
    n_tot = overlap_rows['direction_concordant'].notna().sum()
    ax.text(0.98, 0.02, f"Direction concordance: {n_conc}/{n_tot} ({n_conc/max(n_tot,1)*100:.0f}%)",
            transform=ax.transAxes, ha='right', va='bottom', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    fig.tight_layout()
    fig.savefig(os.path.join(OVERLAP_FIG_PATH, 'track1_cohort_hr_concordance.png'))
    plt.show()
else:
    print("No overlapping markers between cohort1 and cohort2 in Track 1.")

## 2. Track 1: ATE vs Unweighted

For each (cohort, ps_model, cancer_type), compare markers significant under ATE generalizability weighting vs no weighting.

In [ ]:
# ── Track 1: ATE vs unweighted overlap ────────────────────────────────

t1_weight_overlap = compute_overlap(
    t1, group_col='weight_type', val_a='ATE', val_b='unweighted',
    fixed_cols=['cohort', 'ps_model', 'cancer_type'],
    hr_col='HR_marker',
)

print("=" * 60)
print("Track 1: ATE (generalizability) vs unweighted")
print("=" * 60)

venn_data_t1_weight = {}
for (coh, ps, ct), grp in t1_weight_overlap.groupby(['cohort', 'ps_model', 'cancer_type']):
    label = f"{coh} / {ps} / {ct}"
    print(f"\n--- {label} ---")
    counts = overlap_summary(grp, 'ATE', 'unweighted')
    venn_data_t1_weight[(coh, ps, ct)] = counts

print(f"\n{'=' * 60}")
print("AGGREGATE")
print("=" * 60)
_ = overlap_summary(t1_weight_overlap, 'ATE', 'unweighted')

In [ ]:
# ── Venn diagrams: Track 1 weighting overlap per stratum ──────────────

strata = sorted(venn_data_t1_weight.keys())
n_plots = len(strata)
ncols = min(n_plots, 3)
nrows = int(np.ceil(n_plots / ncols)) if n_plots > 0 else 1

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
if n_plots == 1:
    axes = np.array([axes])
axes = axes.flatten()

for i, (coh, ps, ct) in enumerate(strata):
    ax = axes[i]
    c = venn_data_t1_weight[(coh, ps, ct)]
    v = venn2(
        subsets=(c['n_a_only'], c['n_b_only'], c['n_overlap']),
        set_labels=('ATE', 'unweighted'),
        ax=ax,
    )
    ax.set_title(f"{ct}\n{coh} / {ps}", fontsize=10)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Track 1: ATE vs Unweighted Marker Overlap", fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(OVERLAP_FIG_PATH, 'track1_weight_overlap_venns.png'))
plt.show()

## 3. Track 2: Matched vs Unmatched & ATE vs noIPTW

In [ ]:
# ── Track 2: cohort1 vs cohort2 overlap ───────────────────────────────

t2_cohort_overlap = compute_overlap(
    t2, group_col='cohort', val_a='cohort1', val_b='cohort2',
    fixed_cols=['ps_model', 'weight_type', 'cancer_type'],
    hr_col='HR_markerxICI',
)

print("=" * 60)
print("Track 2: cohort1 (unmatched) vs cohort2 (matched)")
print("=" * 60)

venn_data_t2_cohort = {}
for keys, grp in t2_cohort_overlap.groupby(['ps_model', 'weight_type', 'cancer_type']):
    ps, wt, ct = keys
    label = f"{ps} / {wt} / {ct}"
    print(f"\n--- {label} ---")
    counts = overlap_summary(grp, 'cohort1', 'cohort2')
    venn_data_t2_cohort[keys] = counts

print(f"\n{'=' * 60}")
print("AGGREGATE")
print("=" * 60)
_ = overlap_summary(t2_cohort_overlap, 'cohort1', 'cohort2')

In [ ]:
# ── Track 2: ATE vs noIPTW overlap ────────────────────────────────────

t2_weight_overlap = compute_overlap(
    t2, group_col='weight_type', val_a='ATE', val_b='noIPTW',
    fixed_cols=['cohort', 'ps_model', 'cancer_type'],
    hr_col='HR_markerxICI',
)

print("=" * 60)
print("Track 2: ATE vs noIPTW")
print("=" * 60)

venn_data_t2_weight = {}
for keys, grp in t2_weight_overlap.groupby(['cohort', 'ps_model', 'cancer_type']):
    coh, ps, ct = keys
    label = f"{coh} / {ps} / {ct}"
    print(f"\n--- {label} ---")
    counts = overlap_summary(grp, 'ATE', 'noIPTW')
    venn_data_t2_weight[keys] = counts

print(f"\n{'=' * 60}")
print("AGGREGATE")
print("=" * 60)
_ = overlap_summary(t2_weight_overlap, 'ATE', 'noIPTW')

In [ ]:
# ── Venn grids: Track 2 cohort & weight overlaps ─────────────────────

def plot_venn_grid(venn_data, label_a, label_b, key_labels, title, filename):
    """Plot a grid of Venn diagrams from a {key: counts} dict."""
    strata = sorted(venn_data.keys())
    n_plots = len(strata)
    if n_plots == 0:
        print(f"  No strata to plot for {title}")
        return
    ncols = min(n_plots, 3)
    nrows = int(np.ceil(n_plots / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
    if nrows * ncols == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for i, key in enumerate(strata):
        ax = axes[i]
        c = venn_data[key]
        venn2(
            subsets=(c['n_a_only'], c['n_b_only'], c['n_overlap']),
            set_labels=(label_a, label_b),
            ax=ax,
        )
        ax.set_title(key_labels(key), fontsize=10)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(title, fontweight='bold', y=1.02)
    fig.tight_layout()
    fig.savefig(os.path.join(OVERLAP_FIG_PATH, filename))
    plt.show()


plot_venn_grid(
    venn_data_t2_cohort, 'cohort1\n(unmatched)', 'cohort2\n(matched)',
    key_labels=lambda k: f"{k[2]}\n{k[0]} / {k[1]}",
    title="Track 2: Matched vs Unmatched Marker Overlap",
    filename='track2_cohort_overlap_venns.png',
)

plot_venn_grid(
    venn_data_t2_weight, 'ATE', 'noIPTW',
    key_labels=lambda k: f"{k[2]}\n{k[0]} / {k[1]}",
    title="Track 2: ATE vs noIPTW Marker Overlap",
    filename='track2_weight_overlap_venns.png',
)

## 4. Cross-Specification Robustness Heatmap

For each significant marker, show a binary heatmap of which specifications it survives.
Rows = markers, columns = specifications. Markers that survive more specifications are more robust.

In [ ]:
# ── Cross-specification robustness heatmap ────────────────────────────

def robustness_heatmap(df, spec_cols, marker_col='marker', hr_col=None,
                       title='', filename=None, min_specs=1):
    """Binary heatmap: rows = markers, cols = specifications.

    spec_cols: list of column names that together define a specification.
    """
    df = df.copy()
    df['spec'] = df[spec_cols].astype(str).agg(' / '.join, axis=1)

    # Pivot to marker x spec binary matrix
    pivot = df.pivot_table(index=marker_col, columns='spec',
                           aggfunc='size', fill_value=0)
    pivot = (pivot > 0).astype(int)

    # Filter to markers in at least min_specs specifications
    pivot = pivot[pivot.sum(axis=1) >= min_specs]
    if pivot.empty:
        print(f"  No markers in >= {min_specs} specifications.")
        return pivot

    # Sort: most robust markers first, then alphabetical
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]

    fig_height = max(4, 0.35 * len(pivot))
    fig_width = max(6, 0.9 * len(pivot.columns))
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))

    sns.heatmap(pivot, cmap=['#f0f0f0', '#2166ac'], cbar=False,
                linewidths=0.5, linecolor='white', ax=ax)
    ax.set_title(title, fontweight='bold', pad=12)
    ax.set_xlabel('Specification')
    ax.set_ylabel('Marker')
    ax.tick_params(axis='x', rotation=45)

    # Add count annotation on the right
    for i, marker in enumerate(pivot.index):
        n = pivot.loc[marker].sum()
        ax.text(len(pivot.columns) + 0.1, i + 0.5, f" {n}",
                va='center', fontsize=8, color='#333')

    fig.tight_layout()
    if filename:
        fig.savefig(os.path.join(OVERLAP_FIG_PATH, filename))
    plt.show()

    return pivot


# Track 1: all markers across all specifications
t1_pivot = robustness_heatmap(
    t1,
    spec_cols=['cohort', 'ps_model', 'weight_type', 'cancer_type'],
    title='Track 1: Cross-Specification Robustness\n(pan_cancer only)',
    filename='track1_robustness_heatmap_pan.png',
    min_specs=2,
)

In [ ]:
# ── Track 2: cross-specification robustness heatmap ───────────────────

t2_pivot = robustness_heatmap(
    t2,
    spec_cols=['cohort', 'ps_model', 'weight_type', 'cancer_type'],
    title='Track 2: Cross-Specification Robustness',
    filename='track2_robustness_heatmap.png',
    min_specs=2,
)

## 5. Summary Tables: Overlapping Markers

Export the overlapping marker sets for downstream KM analysis.

In [ ]:
# ── Summary: markers overlapping across cohorts (Track 1) ────────────

t1_cohort_both = t1_cohort_overlap[t1_cohort_overlap['overlap']].copy()
t1_cohort_both = t1_cohort_both.sort_values(
    ['cancer_type', 'ps_model', 'weight_type', 'marker']
)

print(f"Track 1 markers significant in BOTH cohort1 & cohort2: "
      f"{len(t1_cohort_both)} rows, {t1_cohort_both['marker'].nunique()} unique markers\n")

display_cols = ['marker', 'cancer_type', 'ps_model', 'weight_type',
                'HR_cohort1', 'HR_cohort2', 'direction_concordant']
if len(t1_cohort_both) > 0:
    display(t1_cohort_both[display_cols].reset_index(drop=True))
    t1_cohort_both.to_csv(
        os.path.join(COMPILED_DIR, 'track1_cohort_overlap_markers.csv'), index=False)
    print(f"\nSaved to {os.path.join(COMPILED_DIR, 'track1_cohort_overlap_markers.csv')}")

In [ ]:
# ── Summary: markers overlapping across weighting schemes ─────────────

# Track 1: ATE vs unweighted
t1_weight_both = t1_weight_overlap[t1_weight_overlap['overlap']].copy()
t1_weight_both = t1_weight_both.sort_values(
    ['cancer_type', 'cohort', 'ps_model', 'marker']
)

print(f"Track 1 markers significant in BOTH ATE & unweighted: "
      f"{len(t1_weight_both)} rows, {t1_weight_both['marker'].nunique()} unique markers\n")

display_cols_w = ['marker', 'cancer_type', 'cohort', 'ps_model',
                  'HR_ATE', 'HR_unweighted', 'direction_concordant']
if len(t1_weight_both) > 0:
    display(t1_weight_both[display_cols_w].reset_index(drop=True))
    t1_weight_both.to_csv(
        os.path.join(COMPILED_DIR, 'track1_weight_overlap_markers.csv'), index=False)

# Track 2: ATE vs noIPTW
t2_weight_both = t2_weight_overlap[t2_weight_overlap['overlap']].copy()
t2_weight_both = t2_weight_both.sort_values(
    ['cancer_type', 'cohort', 'ps_model', 'marker']
)

print(f"\nTrack 2 markers significant in BOTH ATE & noIPTW: "
      f"{len(t2_weight_both)} rows, {t2_weight_both['marker'].nunique()} unique markers\n")

display_cols_w2 = ['marker', 'cancer_type', 'cohort', 'ps_model',
                   'HR_ATE', 'HR_noIPTW', 'direction_concordant']
if len(t2_weight_both) > 0:
    display(t2_weight_both[display_cols_w2].reset_index(drop=True))
    t2_weight_both.to_csv(
        os.path.join(COMPILED_DIR, 'track2_weight_overlap_markers.csv'), index=False)

# Track 2: cohort overlap
t2_cohort_both = t2_cohort_overlap[t2_cohort_overlap['overlap']].copy()
print(f"\nTrack 2 markers significant in BOTH cohort1 & cohort2: "
      f"{len(t2_cohort_both)} rows, {t2_cohort_both['marker'].nunique()} unique markers")
if len(t2_cohort_both) > 0:
    display(t2_cohort_both.reset_index(drop=True))
    t2_cohort_both.to_csv(
        os.path.join(COMPILED_DIR, 'track2_cohort_overlap_markers.csv'), index=False)

In [ ]:
# ── Collect overlap markers for KM notebook ──────────────────────────

# All unique (marker, cancer_type) tuples that overlap on at least one axis
km_candidates = set()

for _, row in t1_cohort_both.iterrows():
    km_candidates.add((row['marker'], row['cancer_type']))
for _, row in t1_weight_both.iterrows():
    km_candidates.add((row['marker'], row['cancer_type']))
for _, row in t2_weight_both.iterrows():
    km_candidates.add((row['marker'], row['cancer_type']))
for _, row in t2_cohort_both.iterrows():
    km_candidates.add((row['marker'], row['cancer_type']))

km_candidates = sorted(km_candidates)
print(f"Total unique (marker, cancer_type) pairs overlapping on any axis: {len(km_candidates)}")
for m, ct in km_candidates:
    print(f"  {m:25s} {ct}")

# Save for use by KM notebook
km_df = pd.DataFrame(km_candidates, columns=['marker', 'cancer_type'])
km_df.to_csv(os.path.join(COMPILED_DIR, 'overlap_markers_for_km.csv'), index=False)
print(f"\nSaved to {os.path.join(COMPILED_DIR, 'overlap_markers_for_km.csv')}")